# 7.4 调用预训练音源分离模型

本 notebook 演示预训练音源分离模型的完整调用流程：准备短音频输入、检查 Python 包与命令行工具、预取模型 checkpoint、运行分离命令、保存 stem 音频，并生成可复核的表格与诊断图。输入案例包括 `MUSDB18-HQ` test 片段、`xiaohetang` 配套素材和管弦乐配套素材；可听结果保存到 `output_audio/07_4/`。

本节涉及三类对象：`package` 是 Python 依赖包，`checkpoint` 是预训练权重文件，`stem taxonomy` 是模型输出音轨的分类方式。2-stem 通常输出 `vocals/accompaniment`，4-stem 通常输出 `vocals/drums/bass/other`，6-stem 在此基础上增加 `guitar/piano`。


## 兼容性提示

- 请在前面章节已经建立的虚拟环境中运行本章 notebook（推荐 Python 3.11）。部分依赖包在 Python 3.13 上可能出现 `collections.Hashable` 等兼容性问题。
- 首次运行预训练模型时，工具会自动下载 checkpoint，需要一定时间，请耐心等待；本章不给出具体下载大小或耗时预估，因为不同网络与硬件差异较大。
- 若某模型依赖缺失，可将 `ENABLE_OPTIONAL_MODELS` 或对应运行开关设为 `0`，跳过该模型继续学习其余内容。


## 1. 运行开关与路径

本单元设置输出目录、模型缓存目录和运行开关。`RUN_MODEL_SETUP` 控制 checkpoint 预取，`RUN_SEPARATION` 控制是否实际执行分离，`CASE_DURATION` 控制每个案例截取的秒数。


In [ ]:
import csv
import importlib.util
import os
import platform
import re
import subprocess
import sys
import tempfile
import textwrap
from dataclasses import dataclass
from pathlib import Path

# matplotlib/numba 缓存目录：用跨平台的系统临时目录（Windows 没有 /tmp）
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)
OUT_AUDIO_DIR = NOTEBOOK_DIR / "output_audio" / "07_4"
OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = NOTEBOOK_DIR / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR = NOTEBOOK_DIR / "checkpoints" / "07_4_models"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REQUIREMENTS_PATH = NOTEBOOK_DIR / "requirements-pretrained.txt"

# 运行开关：常规执行会准备模型并分离短片段；批量验证可用环境变量关闭。
RUN_MODEL_SETUP = os.environ.get("CHAPTER07_RUN_MODEL_SETUP", "1") != "0"
DOWNLOAD_MISSING_MODELS = os.environ.get("CHAPTER07_DOWNLOAD_MISSING_MODELS", "1") != "0"
RUN_SEPARATION = os.environ.get("CHAPTER07_RUN_SEPARATION", "1") != "0"
INSTALL_MISSING_PACKAGES = os.environ.get("CHAPTER07_INSTALL_MISSING_PACKAGES", "0") == "1"
ENABLE_OPTIONAL_MODELS = os.environ.get("CHAPTER07_ENABLE_OPTIONAL_MODELS", "1") != "0"
REUSE_EXISTING_OUTPUTS = os.environ.get("CHAPTER07_REUSE_EXISTING_OUTPUTS", "1") != "0"

CASE_DURATION = float(os.environ.get("CHAPTER07_CASE_DURATION", "20"))
MAX_MUSDB_CASES = int(os.environ.get("CHAPTER07_MAX_MUSDB_CASES", "2"))
DEVICE = os.environ.get("CHAPTER07_DEVICE", "cpu")

print("Python:", sys.version.split()[0], platform.platform())
print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))
print("output_audio:", OUT_AUDIO_DIR.relative_to(REPO_ROOT))
print("model_cache:", MODEL_CACHE_DIR.relative_to(REPO_ROOT))
print("RUN_MODEL_SETUP:", RUN_MODEL_SETUP)
print("DOWNLOAD_MISSING_MODELS:", DOWNLOAD_MISSING_MODELS)
print("RUN_SEPARATION:", RUN_SEPARATION)
print("INSTALL_MISSING_PACKAGES:", INSTALL_MISSING_PACKAGES)
print("ENABLE_OPTIONAL_MODELS:", ENABLE_OPTIONAL_MODELS)
print()
print("依赖安装提示：")
print("1. 请在前面章节已经建立的虚拟环境中运行本 notebook。")
print("2. 如需在终端安装依赖，请先激活同一个虚拟环境。")
print("3. 依赖包括 PyTorch、Torchaudio、TorchCodec、Demucs、Open-Unmix 与 audio-separator；安装命令取决于终端当前目录：")
print("   - 在项目根目录 Music_AI_Intro 下：python -m pip install -r CODE/chapter07/requirements-pretrained.txt")
print("   - 在 CODE 目录下：python -m pip install -r chapter07/requirements-pretrained.txt")
print("   - 在 CODE/chapter07 目录下：python -m pip install -r requirements-pretrained.txt")
print("4. BS-RoFormer 与 Mel-RoFormer / MelBand RoFormer 通过 audio-separator 调用；Spleeter 是经典模型，只简要说明依赖限制。")
print("5. 在 Jupyter 中运行时，请确认当前 Kernel 指向该虚拟环境。")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from chapter07._common.audio_io import load_audio, save_audio
from chapter07._common.dataset_paths import find_musdb_root, list_musdb_tracks
from chapter07._common.evaluation import pick_evaluation_start_sec
from chapter07._common.external_diagnostics import (
    find_case_mixture_path,
    parse_semicolon_list,
    read_external_manifest,
    repo_relative_path,
    resolve_case_dir,
)
from chapter07._common.metrics import energy_ratio, reconstruction_error
from chapter07._common.plotting import FIGURE_SAVE_DPI, display_label, plot_stem_spectrogram_grid
from chapter07._common.synthesis import make_synthetic_mixture
from chapter07.pretrained import (
    AudioSeparatorRunner,
    DemucsSeparator,
    OpenUnmixSeparator,
    SeparatorUnavailable,
    TorchaudioHDemucsSeparator,
    shell_join,
)


## 2. 工具函数

本单元提供路径显示、命令执行、CSV 写入和表格打印工具。所有外部模型调用都记录为命令行字符串，便于复现实验或在终端单独运行。


In [ ]:
AUDIO_EXTENSIONS = (".wav", ".flac", ".mp3", ".aif", ".aiff", ".ogg")


def slugify(value: str) -> str:
    clean = re.sub(r"[^0-9A-Za-z]+", "_", value).strip("_").lower()
    return clean or "case"


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def short_text(value: str, limit: int = 500) -> str:
    text = " ".join(str(value).split())
    return text[:limit] + (" ..." if len(text) > limit else "")


def run_command(command: list[str], timeout: int | None = None) -> dict[str, object]:
    try:
        completed = subprocess.run(
            command,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        return {
            "returncode": completed.returncode,
            "stdout": short_text(completed.stdout),
            "stderr": short_text(completed.stderr),
        }
    except Exception as exc:
        return {"returncode": -1, "stdout": "", "stderr": f"{type(exc).__name__}: {exc}"}


def display_command(command: list[str]) -> str:
    display_args = []
    for arg in command:
        text = str(arg)
        path = Path(text)
        if path.is_absolute():
            text = repo_relative_path(path, REPO_ROOT)
        display_args.append(text)
    return shell_join(display_args)


def missing_dependency_message(spec: object, status: object) -> str:
    missing = []
    if not package_available(spec.required_package):
        missing.append(f"{spec.required_package} 未安装")
    if spec.requires_command and not status.command_available:
        command_name = getattr(spec.separator, "command_name", "")
        if command_name:
            missing.append(f"{command_name} 命令不可用")
    missing_text = "、".join(missing) if missing else "依赖"
    stable_packages = {"demucs", "openunmix", "torchaudio", "audio_separator"}
    if not spec.core and spec.required_package not in stable_packages:
        return (
            f"{spec.model_id} 尚未准备：{missing_text}。"
            "这是可选扩展模型；当前环境可直接跳过。兼容环境说明见 requirements-pretrained-optional.txt。"
        )
    return (
        f"{spec.model_id} 尚未准备：{missing_text}。"
        "请先激活前面章节建立的虚拟环境，并安装 requirements-pretrained.txt。"
    )


def write_rows(path: Path, rows: list[dict[str, object]], fieldnames: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not fieldnames:
        keys = []
        for row in rows:
            for key in row:
                if key not in keys:
                    keys.append(key)
        fieldnames = keys
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def print_rows(rows: list[dict[str, object]], columns: list[str]) -> None:
    if not rows:
        print("(no rows)")
        return
    widths = {
        col: min(max(len(str(row.get(col, ""))) for row in rows + [{col: col}]), 42)
        for col in columns
    }
    print(" | ".join(col.ljust(widths[col]) for col in columns))
    print("-+-".join("-" * widths[col] for col in columns))
    for row in rows:
        cells = []
        for col in columns:
            text = str(row.get(col, ""))
            if len(text) > widths[col]:
                text = text[: widths[col] - 3] + "..."
            cells.append(text.ljust(widths[col]))
        print(" | ".join(cells))


ENERGY_TOP_N = int(os.environ.get("CHAPTER07_ENERGY_TOP_N", "40"))
ENERGY_TOP_N_PER_CASE = int(os.environ.get("CHAPTER07_ENERGY_TOP_N_PER_CASE", "28"))
ENERGY_LABEL_WIDTH = int(os.environ.get("CHAPTER07_ENERGY_LABEL_WIDTH", "64"))


def compact_label(value: str, width: int = ENERGY_LABEL_WIDTH) -> str:
    text = " ".join(str(value).split())
    return textwrap.shorten(text, width=width, placeholder="...")


def plot_energy_rows(
    rows: list[dict[str, object]],
    out_path: Path,
    title: str,
    top_n: int,
    include_case: bool,
) -> plt.Figure | None:
    if not rows:
        return None
    plot_rows = sorted(rows, key=lambda row: float(row["energy_ratio"]), reverse=True)[:top_n]
    labels = []
    for row in plot_rows:
        stem_label = display_label(str(row["stem"]))
        model_label = row.get("model_display_name", row["model_id"])
        if include_case:
            raw_label = f"{row['case_display_name']} | {model_label} | {stem_label}"
        else:
            raw_label = f"{model_label} | {stem_label}"
        labels.append(compact_label(raw_label))

    values = [float(row["energy_ratio"]) for row in plot_rows]
    height = max(4.5, 0.34 * len(plot_rows) + 1.6)
    fig, ax = plt.subplots(figsize=(11, height))
    y_pos = np.arange(len(plot_rows))
    ax.barh(y_pos, values, color="0.35")
    ax.set_yticks(y_pos, labels)
    ax.invert_yaxis()
    ax.set_xlabel("能量比")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
    return fig


## 3. 准备输入案例

`SeparationCase` 描述一个待分离音频片段，包括来源文件、截取起点、截取长度和预期 stem 标签。`expected_present`、`expected_absent`、`expected_uncertain` 用于后续无参考诊断，不等同于严格的监督标注。


### 关于片段起点选择

默认对 MUSDB18-HQ 曲目使用基于能量的片段选择：在整首曲目上滑动等长窗口，挑选总能量最高且各参考 stem 均不低于 -60 dBFS 的窗口作为 `start_sec`。这可以避免把评估建在前奏、休止或某件乐器几乎无声的段落上，从而减少 SI-SDR 出现极端负值的误导情况。

需要注意：
- **能量只是代理指标**。高能量窗口并不保证所有乐器都在活跃，低能量窗口也不代表模型错误。
- 如果希望严格评估，可把 `CASE_DURATION` 设得很大或修改 `select_musdb_cases` 直接分离全曲；全曲评估更准确，但会显著增加运行时间。
- 配套素材与合成案例仍沿用各自的 `start_sec` 设置。


In [ ]:
@dataclass
class SeparationCase:
    case_id: str
    display_name: str
    family: str
    source_path: Path
    start_sec: float
    duration_sec: float
    expected_present: tuple[str, ...] = ()
    expected_absent: tuple[str, ...] = ()
    expected_uncertain: tuple[str, ...] = ()
    notes: str = ""
    input_path: Path | None = None


def select_musdb_cases(limit: int = 2) -> list[SeparationCase]:
    root = find_musdb_root()
    if root is None:
        print("未找到 MUSDB18-HQ。可设置 MUSDB18HQ_ROOT，或放到 CODE/datasets/musdb18-hq。")
        return []
    for split in ("test", "valid", "train"):
        tracks = list_musdb_tracks(root, split=split)
        if tracks:
            selected = tracks[:limit]
            print(f"MUSDB18-HQ root: {repo_relative_path(root, REPO_ROOT)} split={split} tracks={len(tracks)}")
            return [
                SeparationCase(
                    case_id=f"musdb_{slugify(track.name)}",
                    display_name=track.name,
                    family="musdb18hq",
                    source_path=track / "mixture.wav",
                    start_sec=pick_evaluation_start_sec(
                        track,
                        duration=CASE_DURATION,
                        hop=5.0,
                        min_stem_energy_db=-60.0,
                        fallback_start_sec=0.0,
                    ),
                    duration_sec=CASE_DURATION,
                    expected_present=("vocals", "drums", "bass", "other"),
                    notes=f"MUSDB18-HQ {split}/{track.name}; start chosen by energy-based segment selection",
                )
                for track in selected
            ]
    print("已找到 MUSDB18-HQ 目录，但没有检测到包含 mixture.wav 的曲目。")
    return []


def select_author_cases() -> list[SeparationCase]:
    manifest_path = NOTEBOOK_DIR / "data_manifests" / "external_cases.csv"
    if not manifest_path.exists():
        manifest_path = NOTEBOOK_DIR / "data_manifests" / "external_cases.example.csv"
    cases = []
    for row in read_external_manifest(manifest_path):
        case_id = row.get("case_id", "").strip()
        if not case_id:
            continue
        case_dir = resolve_case_dir(row.get("case_dir", ""), REPO_ROOT)
        mixture_path = find_case_mixture_path(case_dir, row, repo_root=REPO_ROOT)
        if mixture_path is None or not mixture_path.exists():
            print(f"跳过 {case_id}：未在 {repo_relative_path(case_dir, REPO_ROOT)} 下找到混音文件。")
            continue
        cases.append(
            SeparationCase(
                case_id=case_id,
                display_name=row.get("title", "").strip() or case_id,
                family=row.get("category", "author").strip() or "author",
                source_path=mixture_path,
                start_sec=float(row.get("start_sec") or 0.0),
                duration_sec=float(row.get("duration_sec") or CASE_DURATION),
                expected_present=tuple(parse_semicolon_list(row.get("expected_present"))),
                expected_absent=tuple(parse_semicolon_list(row.get("expected_absent"))),
                expected_uncertain=tuple(parse_semicolon_list(row.get("expected_uncertain"))),
                notes=row.get("notes", ""),
            )
        )
    return cases


def make_synthetic_fallback() -> list[SeparationCase]:
    fallback_dir = OUT_AUDIO_DIR / "synthetic_fallback"
    fallback_dir.mkdir(parents=True, exist_ok=True)
    path = fallback_dir / "mixture.wav"
    if not path.exists():
        sources = make_synthetic_mixture(sr=22050, duration=8.0, seed=74)
        save_audio(path, sources["mixture"], 22050)
    return [
        SeparationCase(
            case_id="synthetic_fallback",
            display_name="synthetic_fallback",
            family="synthetic",
            source_path=path,
            start_sec=0.0,
            duration_sec=8.0,
            notes="MUSDB18-HQ 与配套素材均不可用时使用的合成案例。",
        )
    ]


cases = select_musdb_cases(MAX_MUSDB_CASES) + select_author_cases()
if not cases:
    cases = make_synthetic_fallback()

case_rows = [
    {
        "case_id": case.case_id,
        "display_name": case.display_name,
        "family": case.family,
        "source": repo_relative_path(case.source_path, REPO_ROOT),
        "start_sec": case.start_sec,
        "duration_sec": case.duration_sec,
        "expected_present": ";".join(case.expected_present),
        "expected_absent": ";".join(case.expected_absent),
        "expected_uncertain": ";".join(case.expected_uncertain),
    }
    for case in cases
]
write_rows(TABLE_DIR / "07_4_input_cases.csv", case_rows)
print_rows(case_rows, ["case_id", "display_name", "family", "source", "duration_sec"])


In [ ]:
def prepare_case_input(case: SeparationCase) -> SeparationCase:
    out_path = OUT_AUDIO_DIR / case.case_id / "input.wav"
    # Always regenerate the input snippet: start_sec/duration_sec may have
    # changed (e.g. via energy-based segment selection) and reusing an old
    # cropped file would silently mismatch the case manifest.
    audio, sr = load_audio(
        case.source_path,
        sr=None,
        mono=False,
        start=case.start_sec,
        duration=case.duration_sec,
    )
    if audio.size == 0:
        raise ValueError(f"Empty audio after cropping: {case.source_path}")
    save_audio(out_path, audio, sr)
    case.input_path = out_path
    return case


cases = [prepare_case_input(case) for case in cases]
for case in cases:
    assert case.input_path is not None
    audio, sr = load_audio(case.input_path, sr=None, mono=False)
    duration = audio.shape[-1] / sr
    print(case.case_id, repo_relative_path(case.input_path, REPO_ROOT), f"{duration:.2f}s", f"{sr} Hz")


## 4. 模型清单与首次下载

`ModelSpec` 描述一个可调用模型，包括模型名称、输出 stem 类型、依赖包、命令行工具和 checkpoint 预取命令。默认执行矩阵包含 Demucs htdemucs_ft、Demucs mdx_extra_q、Torchaudio HDemucs、Open-Unmix、BS-RoFormer 与 Mel-RoFormer / MelBand RoFormer。Spleeter 作为经典模型进入参考表，记录其依赖限制。

首次运行时，预训练模型通常需要下载 checkpoint。下载位置由各工具自身管理。TorchCodec 用于部分 PyTorch / Torchaudio 音频保存路径，已放入核心依赖文件；audio-separator 的 RoFormer 模型缓存放在 `checkpoints/07_4_models/audio_separator/`。


In [ ]:
@dataclass
class ModelSpec:
    model_id: str
    display_name: str
    separator: object
    mode: str
    required_package: str
    install_command: str
    core: bool
    prefetch_command: list[str] | None
    requires_command: bool = False
    run_on_families: tuple[str, ...] = ("musdb18hq", "pop", "orchestra")
    notes: str = ""


def demucs_prefetch(model_name: str) -> list[str]:
    return [
        sys.executable,
        "-c",
        f"from demucs.pretrained import get_model; get_model({model_name!r}); print('ready:{model_name}')",
    ]


def openunmix_prefetch(model_name: str = "umxhq") -> list[str]:
    return [
        sys.executable,
        "-c",
        (
            "import openunmix; "
            f"openunmix.{model_name}(targets=['vocals','drums','bass','other'], "
            "device='cpu', pretrained=True); "
            f"print('ready:{model_name}')"
        ),
    ]


def torchaudio_prefetch(bundle_name: str = "HDEMUCS_HIGH_MUSDB_PLUS") -> list[str]:
    return [
        sys.executable,
        "-c",
        (
            "import torchaudio; "
            f"bundle = torchaudio.pipelines.{bundle_name}; "
            "bundle.get_model(); "
            f"print('ready:{bundle_name}')"
        ),
    ]


audio_separator_model_dir = MODEL_CACHE_DIR / "audio_separator"
audio_separator_model_dir.mkdir(parents=True, exist_ok=True)
bs_roformer = AudioSeparatorRunner(model_file_dir=audio_separator_model_dir, log_level="warning")
mel_roformer = AudioSeparatorRunner(
    model_filename="melband_roformer_big_beta4.ckpt",
    model_file_dir=audio_separator_model_dir,
    log_level="warning",
)

model_specs = [
    ModelSpec(
        model_id="demucs_htdemucs_ft_4stem",
        display_name="Demucs htdemucs_ft 4-stem",
        separator=DemucsSeparator(model="htdemucs_ft", device=DEVICE),
        mode="4stems",
        required_package="demucs",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=True,
        prefetch_command=demucs_prefetch("htdemucs_ft"),
        notes="Hybrid Transformer Demucs fine-tuned checkpoint; MUSDB-style 4-stem.",
    ),
    ModelSpec(
        model_id="demucs_htdemucs_ft_2stem",
        display_name="Demucs htdemucs_ft 2-stem",
        separator=DemucsSeparator(model="htdemucs_ft", device=DEVICE),
        mode="2stems",
        required_package="demucs",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=True,
        prefetch_command=demucs_prefetch("htdemucs_ft"),
        notes="vocals/accompaniment; practical karaoke use case.",
    ),
    ModelSpec(
        model_id="demucs_htdemucs_6stem",
        display_name="Demucs htdemucs_6s 6-stem",
        separator=DemucsSeparator(model="htdemucs", device=DEVICE),
        mode="6stems",
        required_package="demucs",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=False,
        prefetch_command=demucs_prefetch("htdemucs_6s"),
        notes="Adds guitar/piano; useful for xiaohetang piano and absent-stem checks.",
    ),
    ModelSpec(
        model_id="demucs_mdx_extra_q_4stem",
        display_name="Demucs mdx_extra_q 4-stem",
        separator=DemucsSeparator(model="mdx_extra_q", device=DEVICE),
        mode="4stems",
        required_package="demucs",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=False,
        prefetch_command=demucs_prefetch("mdx_extra_q"),
        notes="MDX-style Demucs checkpoint; lighter comparison against htdemucs_ft.",
    ),
    ModelSpec(
        model_id="torchaudio_hdemucs_high_musdb_plus",
        display_name="Torchaudio HDemucs High MUSDB+",
        separator=TorchaudioHDemucsSeparator(device=DEVICE),
        mode="4stems",
        required_package="torchaudio",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=True,
        prefetch_command=torchaudio_prefetch("HDEMUCS_HIGH_MUSDB_PLUS"),
        notes="Official PyTorch/Torchaudio source-separation bundle.",
    ),
    ModelSpec(
        model_id="openunmix_umxhq_4stem",
        display_name="Open-Unmix umxhq 4-stem",
        separator=OpenUnmixSeparator(model="umxhq", no_cuda=(DEVICE == "cpu")),
        mode="4stems",
        required_package="openunmix",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=True,
        prefetch_command=openunmix_prefetch("umxhq"),
        requires_command=False,
        notes="Classic STFT magnitude + BLSTM + Wiener filtering baseline; uses a local CLI wrapper.",
    ),
    ModelSpec(
        model_id="bs_roformer_audio_separator",
        display_name="BS-RoFormer via audio-separator",
        separator=bs_roformer,
        mode="2stems",
        required_package="audio_separator",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=False,
        prefetch_command=bs_roformer.build_download_command(),
        requires_command=False,
        notes="RoFormer-family checkpoint, usually vocals/accompaniment; uses a local CLI wrapper.",
    ),
    ModelSpec(
        model_id="melband_roformer_big_beta4",
        display_name="Mel-RoFormer / MelBand RoFormer Big Beta 4",
        separator=mel_roformer,
        mode="2stems",
        required_package="audio_separator",
        install_command="python -m pip install -r requirements-pretrained.txt",
        core=False,
        prefetch_command=mel_roformer.build_download_command(),
        requires_command=False,
        notes="Mel-band RoFormer checkpoint; vocals/other target with mel-scale band modeling; uses a local CLI wrapper.",
    ),
]

if not ENABLE_OPTIONAL_MODELS:
    model_specs = [spec for spec in model_specs if spec.core]

core_dependency_rows = [
    {"dependency": name, "available": package_available(name)}
    for name in ("torch", "torchaudio", "torchcodec", "demucs", "openunmix", "audio_separator")
]
write_rows(TABLE_DIR / "07_4_core_dependency_status.csv", core_dependency_rows)
print_rows(core_dependency_rows, ["dependency", "available"])

print_rows(
    [
        {
            "model_id": spec.model_id,
            "core": spec.core,
            "package": spec.required_package,
            "mode": spec.mode,
            "notes": spec.notes,
        }
        for spec in model_specs
    ],
    ["model_id", "core", "package", "mode"],
)

reference_model_rows = [
    {
        "model_id": "spleeter_2stem",
        "role": "historical_u_net_baseline",
        "default_run": False,
        "dependency": "spleeter",
        "limitation": "Older TensorFlow stack; often fails on recent Python or NumPy.",
    },
]
write_rows(TABLE_DIR / "07_4_reference_optional_models.csv", reference_model_rows)
print_rows(reference_model_rows, ["model_id", "role", "dependency", "limitation"])


In [ ]:
def install_missing_package(spec: ModelSpec) -> dict[str, object]:
    command = [sys.executable, "-m", "pip", "install", spec.required_package.replace("_", "-")]
    print(f"正在安装缺失依赖：{spec.required_package}")
    print(display_command(command))
    return run_command(command, timeout=None)


model_setup_rows = []
model_ready = {}

for spec in model_specs:
    status = spec.separator.dependency_status()
    if INSTALL_MISSING_PACKAGES and not package_available(spec.required_package):
        install_result = install_missing_package(spec)
        status = spec.separator.dependency_status()
    else:
        install_result = {"returncode": "", "stdout": "", "stderr": ""}

    package_ok = package_available(spec.required_package)
    command_ok = bool(status.command_available)
    dependency_ok = package_ok and (command_ok or not spec.requires_command)
    download_result = {"returncode": "", "stdout": "", "stderr": ""}

    if RUN_MODEL_SETUP and DOWNLOAD_MISSING_MODELS and dependency_ok and spec.prefetch_command:
        print("准备模型 checkpoint:", spec.model_id)
        print(display_command(spec.prefetch_command))
        download_result = run_command(spec.prefetch_command, timeout=None)
    elif not dependency_ok:
        print(missing_dependency_message(spec, status))

    ready = dependency_ok and (download_result["returncode"] in ("", 0))
    model_ready[spec.model_id] = ready
    model_setup_rows.append(
        {
            "model_id": spec.model_id,
            "display_name": spec.display_name,
            "core": spec.core,
            "package_available": package_ok,
            "command_available": command_ok,
            "dependency_ready": dependency_ok,
            "model_ready": ready,
            "install_command": spec.install_command,
            "prefetch_command": display_command(spec.prefetch_command) if spec.prefetch_command else "",
            "prefetch_returncode": download_result["returncode"],
            "prefetch_stderr": download_result["stderr"],
            "notes": spec.notes,
        }
    )

write_rows(TABLE_DIR / "07_4_model_setup.csv", model_setup_rows)
print_rows(
    model_setup_rows,
    ["model_id", "dependency_ready", "model_ready", "prefetch_returncode"],
)


## 5. 运行分离并保存可听结果

本单元对每个输入案例运行已准备好的模型，并把输出 stem 保存到 `output_audio/07_4/<case_id>/<model_id>/`。命令、状态、输出目录和 stem 清单写入 `07_4_pretrained_commands.csv`。


In [ ]:
run_rows = []
results = {}

for case in cases:
    assert case.input_path is not None
    for spec in model_specs:
        out_dir = OUT_AUDIO_DIR / case.case_id / spec.model_id
        command = spec.separator.build_command(case.input_path, out_dir, stems=spec.mode)
        run_row = {
            "case_id": case.case_id,
            "family": case.family,
            "model_id": spec.model_id,
            "mode": spec.mode,
            "input_path": repo_relative_path(case.input_path, REPO_ROOT),
            "output_dir": repo_relative_path(out_dir, REPO_ROOT),
            "command": display_command(command),
            "status": "pending",
            "stems": "",
            "notes": "",
        }

        if case.family not in spec.run_on_families:
            run_row["status"] = "skipped_case_family"
            run_rows.append(run_row)
            continue
        if not model_ready.get(spec.model_id, False):
            run_row["status"] = "skipped_model_not_ready"
            run_row["notes"] = "See outputs/tables/07_4_model_setup.csv"
            run_rows.append(run_row)
            continue

        existing = spec.separator.collect_result(case.input_path, out_dir)
        input_mtime = case.input_path.stat().st_mtime
        output_mtimes = [path.stat().st_mtime for path in existing.stems.values() if path.exists()]
        outputs_outdated = output_mtimes and min(output_mtimes) < input_mtime
        if REUSE_EXISTING_OUTPUTS and existing.stems and not outputs_outdated:
            result = existing
            run_row["status"] = "reused_existing_outputs"
        elif not RUN_SEPARATION:
            run_row["status"] = "skipped_run_separation_false"
            run_rows.append(run_row)
            continue
        else:
            try:
                result = spec.separator.separate(case.input_path, out_dir, stems=spec.mode)
                run_row["status"] = "completed"
            except (SeparatorUnavailable, subprocess.CalledProcessError, RuntimeError, OSError) as exc:
                run_row["status"] = "failed"
                run_row["notes"] = f"{type(exc).__name__}: {short_text(exc)}"
                run_rows.append(run_row)
                continue

        results[(case.case_id, spec.model_id)] = result
        run_row["stems"] = ";".join(sorted(result.stems))
        run_rows.append(run_row)
        print(case.case_id, spec.model_id, run_row["status"], run_row["stems"])

write_rows(TABLE_DIR / "07_4_pretrained_commands.csv", run_rows)
print_rows(run_rows, ["case_id", "model_id", "status", "stems"])


In [ ]:
audio_rows = []
for (case_id, model_id), result in sorted(results.items()):
    for stem, path in sorted(result.stems.items()):
        audio_rows.append(
            {
                "case_id": case_id,
                "model_id": model_id,
                "stem": stem,
                "path": repo_relative_path(path, REPO_ROOT),
                "sample_rate": result.sample_rate,
            }
        )

write_rows(TABLE_DIR / "07_4_output_audio_manifest.csv", audio_rows)
print_rows(audio_rows, ["case_id", "model_id", "stem", "path"])


## 6. 频谱、能量与重建诊断

频谱图用于观察不同 stem 的时频能量分布，子图标题按“曲目/案例名称、模型、stem”组织。`energy_ratio` 表示某个 stem 相对 mixture 的能量比例，`reconstruction_error` 表示分离 stem 求和后与 mixture 的差异；这些指标用于快速定位泄漏、残留和输出异常。能量柱状图采用横向标签，完整明细写入 CSV。


In [ ]:
energy_rows = []

for case in cases:
    assert case.input_path is not None
    mixture, sr = load_audio(case.input_path, sr=22050, mono=True)
    for spec in model_specs:
        result = results.get((case.case_id, spec.model_id))
        if result is None:
            continue
        stems_audio = {}
        for stem, path in sorted(result.stems.items()):
            y, _ = load_audio(path, sr=22050, mono=True, duration=case.duration_sec)
            stems_audio[stem] = y
        if not stems_audio:
            continue

        fig_path = FIG_DIR / f"07_4_{case.case_id}_{spec.model_id}_spectrograms.png"
        titled_stems_audio = {
            f"{case.display_name}\n{spec.display_name} | {display_label(stem)}": y
            for stem, y in stems_audio.items()
        }
        fig = plot_stem_spectrogram_grid(titled_stems_audio, 22050, fig_path)
        plt.show()
        plt.close(fig)

        try:
            recon = reconstruction_error(mixture, stems_audio)
        except ValueError:
            recon = np.nan
        for stem, y in stems_audio.items():
            ratio = energy_ratio(y, mixture)
            energy_rows.append(
                {
                    "case_id": case.case_id,
                    "case_display_name": case.display_name,
                    "family": case.family,
                    "model_id": spec.model_id,
                    "model_display_name": spec.display_name,
                    "mode": spec.mode,
                    "stem": stem,
                    "energy_ratio": ratio,
                    "reconstruction_error": recon,
                    "expected_absent": stem in case.expected_absent,
                    "expected_uncertain": stem in case.expected_uncertain,
                    "expected_present": stem in case.expected_present,
                }
            )

if energy_rows:
    write_rows(TABLE_DIR / "07_4_energy_summary.csv", energy_rows)
    fig = plot_energy_rows(
        energy_rows,
        FIG_DIR / "07_4_model_comparison_energy.png",
        f"整体 stem 能量比排序（Top {ENERGY_TOP_N}）",
        top_n=ENERGY_TOP_N,
        include_case=True,
    )
    if fig is not None:
        plt.show()
        plt.close(fig)
    for case in cases:
        case_rows_for_plot = [row for row in energy_rows if row["case_id"] == case.case_id]
        fig = plot_energy_rows(
            case_rows_for_plot,
            FIG_DIR / f"07_4_{case.case_id}_energy_ratios.png",
            f"{case.display_name}：各模型 stem 能量比（Top {ENERGY_TOP_N_PER_CASE}）",
            top_n=ENERGY_TOP_N_PER_CASE,
            include_case=False,
        )
        if fig is not None:
            plt.show()
            plt.close(fig)
    print_rows(energy_rows, ["case_id", "case_display_name", "model_id", "model_display_name", "stem", "energy_ratio", "reconstruction_error"])
else:
    print("尚未生成分离音频。请查看 07_4_model_setup.csv 与 07_4_pretrained_commands.csv。")


## 7. 试听输出

本单元在 notebook 中嵌入输入片段与分离结果播放器。完整音频文件清单保存在 `07_4_output_audio_manifest.csv`，可在文件系统中直接打开对应 wav 文件。


In [ ]:
MAX_AUDIO_PLAYERS = int(os.environ.get("CHAPTER07_MAX_AUDIO_PLAYERS", "24"))
shown = 0
for case in cases:
    if shown >= MAX_AUDIO_PLAYERS:
        break
    assert case.input_path is not None
    print(f"INPUT {case.case_id}: {repo_relative_path(case.input_path, REPO_ROOT)}")
    display(Audio(filename=str(case.input_path)))
    shown += 1
    for spec in model_specs:
        if shown >= MAX_AUDIO_PLAYERS:
            break
        result = results.get((case.case_id, spec.model_id))
        if result is None:
            continue
        for stem, path in sorted(result.stems.items()):
            if shown >= MAX_AUDIO_PLAYERS:
                break
            print(f"{case.case_id} | {spec.model_id} | {stem}: {repo_relative_path(path, REPO_ROOT)}")
            display(Audio(filename=str(path)))
            shown += 1

if shown == 0:
    print("尚无模型输出，因此未显示音频播放器。")
elif shown >= MAX_AUDIO_PLAYERS:
    print(f"已显示前 {MAX_AUDIO_PLAYERS} 个播放器。完整文件清单见 07_4_output_audio_manifest.csv。")


## 8. 结果解读要点

- MUSDB18-HQ 对齐 `vocals/drums/bass/other` 四轨任务定义，可检查 stem 质量、能量分布与重建误差。
- xiaohetang 案例用于观察人声/伴奏分离、钢琴 stem 输出和伴奏泄漏。这里的钢琴可能带混响、压缩或合成器式音色，也可能被其他声部遮蔽。
- 管弦乐案例用于观察流行音乐 stem taxonomy 与工程乐器组之间的映射差异。当前 0--30 秒片段的 brass/choirs 参考近似静默，percussion 则有活动；缺席或不确定标签均按该区间的参考状态解释。
- 2-stem、4-stem、6-stem 使用不同 stem taxonomy；结果按各自任务定义分别解读。
- BS-RoFormer 与 Mel-RoFormer / MelBand RoFormer 都通过 audio-separator 调用。前者强调频带分块后的 Transformer 建模，后者把频带组织进一步贴近 mel 频率尺度；二者可与 Demucs/Open-Unmix 的输出形态对照。
- Spleeter 表示历史 U-Net 路线，仅在参考表中说明依赖限制。
